In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
import json
import pickle
import joblib
import os

In [3]:
# Load data
df = pd.read_csv('../data/features/engineered_features.csv')
selected_features = pd.read_json('../data/selected/selected_features.json')['features'].tolist()

In [4]:
# Load target
TARGET = 'Lowest_20%_Petrol, diesel and oils'
y = df[TARGET]

# Create X with selected features
X = df[selected_features]

print(f"X: {X.shape}")
print(f"y: {y.shape}")
print(f"Selected features: {len(selected_features)}")

X: (69, 13)
y: (69,)
Selected features: 13


In [5]:
# Choose Splitting Strategy
print("CHOOSING SPLITTING STRATEGY")

# Random Split (for general ML)
def random_split(X, y, train_size=0.6, val_size=0.2, test_size=0.2, random_state=42):
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(val_size + test_size), random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(test_size/(val_size+test_size)), random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

# Time Series Split (for chronological data)
def time_split(X, y, train_size=0.6, val_size=0.2, test_size=0.2):
    n = len(X)
    train_end = int(n * train_size)
    val_end = int(n * (train_size + val_size))
    
    X_train = X.iloc[:train_end]
    X_val = X.iloc[train_end:val_end]
    X_test = X.iloc[val_end:]
    
    y_train = y.iloc[:train_end]
    y_val = y.iloc[train_end:val_end]
    y_test = y.iloc[val_end:]
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Choose split strategy
SPLIT_STRATEGY = 'time_series'  # or 'random'
print(f"Using: {SPLIT_STRATEGY} split")

if SPLIT_STRATEGY == 'time_series':
    X_train, X_val, X_test, y_train, y_val, y_test = time_split(X, y)
else:
    X_train, X_val, X_test, y_train, y_val, y_test = random_split(X, y)

print(f"\nSplit sizes:")
print(f"  Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Val: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")


CHOOSING SPLITTING STRATEGY
Using: time_series split

Split sizes:
  Train: 41 (59.4%)
  Val: 14 (20.3%)
  Test: 14 (20.3%)


In [6]:
# Validate Splits
print("VALIDATING SPLITS")

# Check distribution statistics
print("\nTarget distribution:")
print(f"  Train: mean={y_train.mean():.4f}, std={y_train.std():.4f}")
print(f"  Val: mean={y_val.mean():.4f}, std={y_val.std():.4f}")
print(f"  Test: mean={y_test.mean():.4f}, std={y_test.std():.4f}")

# Check for data leakage (if time series)
if SPLIT_STRATEGY == 'time_series':
    print("\nTime series check:")
    print(f"  Train ends: {df['year_month'].iloc[len(X_train)-1]}")
    print(f"  Val starts: {df['year_month'].iloc[len(X_train)]}")
    print(f"  Val ends: {df['year_month'].iloc[len(X_train)+len(X_val)-1]}")
    print(f"  Test starts: {df['year_month'].iloc[len(X_train)+len(X_val)]}")
    
    # Ensure no chronological overlap
    if (df['year_month'].iloc[len(X_train)-1] < df['year_month'].iloc[len(X_train)]):
        print("  ✅ Chronological order maintained")

VALIDATING SPLITS

Target distribution:
  Train: mean=10.9115, std=0.4117
  Val: mean=11.1900, std=0.2876
  Test: mean=10.9443, std=0.0668

Time series check:
  Train ends: 2024-04
  Val starts: 2024-05
  Val ends: 2025-06
  Test starts: 2025-07
  ✅ Chronological order maintained


In [7]:
# Scale Features
print("SCALING FEATURES...")

# Fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled shapes:")
print(f"  X_train: {X_train_scaled.shape}")
print(f"  X_val: {X_val_scaled.shape}")
print(f"  X_test: {X_test_scaled.shape}")

SCALING FEATURES...
Scaled shapes:
  X_train: (41, 13)
  X_val: (14, 13)
  X_test: (14, 13)


In [8]:
# Save Splits
print("SAVING SPLITS...")

os.makedirs('../data/splits', exist_ok=True)

# Save as CSV
X_train.to_csv('../data/splits/X_train.csv', index=False)
X_val.to_csv('../data/splits/X_val.csv', index=False)
X_test.to_csv('../data/splits/X_test.csv', index=False)

pd.DataFrame(y_train, columns=['target']).to_csv('../data/splits/y_train.csv', index=False)
pd.DataFrame(y_val, columns=['target']).to_csv('../data/splits/y_val.csv', index=False)
pd.DataFrame(y_test, columns=['target']).to_csv('../data/splits/y_test.csv', index=False)

# Save scaled versions
pd.DataFrame(X_train_scaled).to_csv('../data/splits/X_train_scaled.csv', index=False)
pd.DataFrame(X_val_scaled).to_csv('../data/splits/X_val_scaled.csv', index=False)
pd.DataFrame(X_test_scaled).to_csv('../data/splits/X_test_scaled.csv', index=False)

# Save as pickle
split_data = {
    'X_train': X_train_scaled,
    'X_val': X_val_scaled,
    'X_test': X_test_scaled,
    'y_train': y_train.values,
    'y_val': y_val.values,
    'y_test': y_test.values
}

with open('../data/splits/split_data.pkl', 'wb') as f:
    pickle.dump(split_data, f)


SAVING SPLITS...


In [9]:
# Save Metadata
print("SAVING SPLIT METADATA...")

split_metadata = {
    'strategy': SPLIT_STRATEGY,
    'splits': {
        'train': {
            'size': len(X_train),
            'percentage': len(X_train)/len(X)*100,
            'date_range': [df['year_month'].iloc[0], df['year_month'].iloc[len(X_train)-1]]
        },
        'validation': {
            'size': len(X_val),
            'percentage': len(X_val)/len(X)*100,
            'date_range': [df['year_month'].iloc[len(X_train)], df['year_month'].iloc[len(X_train)+len(X_val)-1]]
        },
        'test': {
            'size': len(X_test),
            'percentage': len(X_test)/len(X)*100,
            'date_range': [df['year_month'].iloc[len(X_train)+len(X_val)], df['year_month'].iloc[-1]]
        }
    },
    'features': {
        'count': len(selected_features),
        'list': selected_features
    },
    'target': TARGET,
    'feature_names': X.columns.tolist(),
    'timestamp': pd.Timestamp.now().isoformat()
}

with open('../data/splits/split_metadata.json', 'w') as f:
    json.dump(split_metadata, f, indent=2)

print(f"   Splits saved to: ../data/splits/")

SAVING SPLIT METADATA...
   Splits saved to: ../data/splits/
